In [1]:
import pandas as pd

df = pd.read_excel("DF_FINAL.xlsx")

df.head()

calles_base = [
        "Avenida Cardenal Herrera Oria",
        "Avenida Alfonso XIII",
        "Avenida Brasilia",
        "Avenida General Perón",
        "Calle Arturo Soria",
        "Calle Costa Rica",
        "Calle Génova",
        "Calle Hermanos García Noblejas",
        "Calle José Abascal",
        "Calle López de Hoyos",
        "Calle Príncipe de Vergara",
        "Calle Serrano",
        "Calle Velázquez",
        "Paseo de la Castellana"
]

calles_noreste = []

for base in calles_base:
    for col in df.columns:
        if base in col:
            calles_noreste.append(col)

L = 3  # número de lags

X = pd.DataFrame()

for street in calles_noreste:
    for lag in range(1, L + 1):
        X[f"{street}_t-{lag}"] = df[street].shift(lag)

# Variables externas reales
X["AEMET_tmed"] = df["AEMET_tmed"]
X["AEMET_prec"] = df["AEMET_prec"]
X["dia_semana"] = df["dia_semana"]
X["festivo"] = df["festivo"]

# Convertir variables categóricas a numéricas
X["dia_semana"] = X["dia_semana"].map({
    "lunes": 0,
    "martes": 1,
    "miércoles": 2,
    "jueves": 3,
    "viernes": 4,
    "sábado": 5,
    "domingo": 6
})

X["festivo"] = X["festivo"].map({
    "no": 0,
    "sí": 1
})


y_col = [col for col in df.columns if "Paseo de la Castellana" in col][0]
y = df[y_col]

# Limpiar NaNs
X = X.dropna()
y = y.loc[X.index]



In [2]:
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]

y_train = y.iloc[:split]
y_test  = y.iloc[split:]

In [3]:
X_train

,Avenida Cardenal Herrera Oria_E-O_t-1,Avenida Cardenal Herrera Oria_E-O_t-2,Avenida Cardenal Herrera Oria_E-O_t-3,Avenida Cardenal Herrera Oria_O-E_t-1,Avenida Cardenal Herrera Oria_O-E_t-2,Avenida Cardenal Herrera Oria_O-E_t-3,Avenida Alfonso XIII_N-S_t-1,Avenida Alfonso XIII_N-S_t-2,Avenida Alfonso XIII_N-S_t-3,Avenida Alfonso XIII_S-N_t-1,...,Paseo de la Castellana_N-S_t-1,Paseo de la Castellana_N-S_t-2,Paseo de la Castellana_N-S_t-3,Paseo de la Castellana_S-N_t-1,Paseo de la Castellana_S-N_t-2,Paseo de la Castellana_S-N_t-3,AEMET_tmed,AEMET_prec,dia_semana,festivo
3,23.0,27.0,96.0,15.0,24.0,52.0,58.0,77.0,119.0,32.0,...,154.0,240.0,340.0,194.0,297.0,516.0,19.6,0.0,4.0,0.0
4,19.0,23.0,27.0,18.0,15.0,24.0,40.0,58.0,77.0,47.0,...,150.0,154.0,240.0,145.0,194.0,297.0,19.6,0.0,4.0,0.0
5,13.0,19.0,23.0,10.0,18.0,15.0,39.0,40.0,58.0,40.0,...,200.0,150.0,154.0,202.0,145.0,194.0,19.6,0.0,4.0,0.0
6,28.0,13.0,19.0,45.0,10.0,18.0,50.0,39.0,40.0,138.0,...,437.0,200.0,150.0,600.0,202.0,145.0,19.6,0.0,4.0,0.0
7,115.0,28.0,13.0,202.0,45.0,10.0,101.0,50.0,39.0,494.0,...,1264.0,437.0,200.0,1682.0,600.0,202.0,19.6,0.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25046,339.0,321.0,307.0,261.0,247.0,278.0,278.0,337.0,273.0,259.0,...,1018.0,866.0,904.0,1138.0,1014.0,942.0,32.2,0.0,4.0,0.0
25047,397.0,339.0,321.0,321.0,261.0,247.0,277.0,278.0,337.0,272.0,...,922.0,1018.0,866.0,906.0,1138.0,1014.0,32.2,0.0,4.0,0.0
25048,353.0,397.0,339.0,187.0,321.0,261.0,253.0,277.0,278.0,194.0,...,746.0,922.0,1018.0,778.0,906.0,1138.0,32.2,0.0,4.0,0.0
25049,230.0,353.0,397.0,182.0,187.0,321.0,202.0,253.0,277.0,159.0,...,716.0,746.0,922.0,750.0,778.0,906.0,32.2,0.0,4.0,0.0


In [4]:
with pd.ExcelWriter("dataset_RF_Castellana.xlsx") as writer:
    X_train.to_excel(writer, sheet_name="X_train")
    X_test.to_excel(writer, sheet_name="X_test")
    y_train.to_excel(writer, sheet_name="y_train")
    y_test.to_excel(writer, sheet_name="y_test")

In [5]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=20,
    random_state=0,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, min_samples_leaf=20, n_estimators=400,
                      n_jobs=-1, random_state=0)

In [6]:
# Predicción en test
y_pred = rf.predict(X_test)

# Predicción próxima hora
X_next = X.iloc[[-1]]
y_next = rf.predict(X_next)

print("Predicción tráfico en Castellana (N-S) próxima hora:", y_next[0])

Predicción tráfico en Castellana (N-O) próxima hora: 269.42047105379305


In [7]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Cálculo de métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

def mape(y_true, y_pred, eps=1e-6):
    return np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100

mape_val = mape(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape_val:.2f} %")

MAE: 66.99
RMSE: 98.49
MAPE: 12.38 %


In [8]:
importances = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances.head(10))

Calle Hermanos García Noblejas_S-N_t-1    0.751130
Calle Costa Rica_E-O_t-1                  0.088861
Calle Arturo Soria_S-N_t-1                0.068907
Paseo de la Castellana_N-S_t-1            0.051469
Calle Príncipe de Vergara_S-N_t-2         0.003172
Calle Arturo Soria_N-S_t-3                0.002706
Calle Génova_E-O_t-2                      0.002057
Paseo de la Castellana_N-S_t-3            0.001537
Avenida Cardenal Herrera Oria_E-O_t-3     0.001258
Avenida Cardenal Herrera Oria_E-O_t-1     0.001089
dtype: float64


In [9]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=0,
    n_jobs=-1,
    scoring="neg_mean_absolute_error"
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

perm_df.head(10)


,feature,importance
45,Calle Hermanos García Noblejas_S-N_t-1,138.012305
69,Paseo de la Castellana_N-S_t-1,127.443661
30,Calle Costa Rica_E-O_t-1,43.181692
27,Calle Arturo Soria_S-N_t-1,14.677172
61,Calle Príncipe de Vergara_S-N_t-2,11.273141
26,Calle Arturo Soria_N-S_t-3,5.297475
37,Calle Génova_E-O_t-2,5.025805
50,Calle José Abascal_O-E_t-3,4.339387
48,Calle José Abascal_O-E_t-1,4.194066
77,dia_semana,2.749302


In [10]:
top_k = 8
top_features = perm_df.head(top_k)
top_features


,feature,importance
45,Calle Hermanos García Noblejas_S-N_t-1,138.012305
69,Paseo de la Castellana_N-S_t-1,127.443661
30,Calle Costa Rica_E-O_t-1,43.181692
27,Calle Arturo Soria_S-N_t-1,14.677172
61,Calle Príncipe de Vergara_S-N_t-2,11.273141
26,Calle Arturo Soria_N-S_t-3,5.297475
37,Calle Génova_E-O_t-2,5.025805
50,Calle José Abascal_O-E_t-3,4.339387


In [11]:
top_streets = (
    top_features["feature"]
    .str.split("_")
    .str[0]
    .value_counts()
)

top_streets

feature
Calle Arturo Soria                2
Paseo de la Castellana            1
Calle Hermanos García Noblejas    1
Calle Costa Rica                  1
Calle Príncipe de Vergara         1
Calle Génova                      1
Calle José Abascal                1
Name: count, dtype: int64